# MCP Trip Tool

this tool uses amadeus api for getting information

In [1]:
#app/mcp_svr/mcp_trip_tools.py
import sys
import os
import json
import asyncio
from typing import Dict, Any, List, Optional
from amadeus import Client, ResponseError
import dotenv
from datetime import datetime, timedelta

dotenv.load_dotenv()

class AmadeusAPI:
    def __init__(self, client_id=None, client_secret=None):
        if client_id is None:
            client_id = os.getenv('AMADEUS_CLIENT_ID')
        if client_secret is None:
            client_secret = os.getenv('AMADEUS_CLIENT_SECRET')
        if not client_id or not client_secret:
            raise ValueError("Client ID and Client Secret must be provided")
        
        self.amadeus = Client(
            client_id=client_id,
            client_secret=client_secret
        )

    def flight_inspiration_search(self, origin, top_k=10):
        try:
            response = self.amadeus.shopping.flight_destinations.get(origin=origin)
            return response.data[:top_k]
        except ResponseError as error:
            return str(error)

    def flight_cheapest_date_search(self, origin, destination, top_k=10):
        try:
            response = self.amadeus.shopping.flight_dates.get(
                origin=origin, 
                destination=destination
            )
            return response.data[:top_k]
        except ResponseError as error:
            return str(error)

    def flight_offers_search(self, origin, destination, departure_date, adults=1, return_date=None, 
                                   max_price=None, currency='USD', travel_class=None, top_k=10):
        try:
            params = {
                'originLocationCode': origin,
                'destinationLocationCode': destination,
                'departureDate': departure_date,
                'adults': adults,
                'currencyCode': currency
            }

            if return_date:
                params['returnDate'] = return_date
            if max_price:
                params['maxPrice'] = max_price
            if travel_class:
                params['travelClass'] = travel_class

            response = self.amadeus.shopping.flight_offers_search.get(**params)
            return response.data[:top_k]
        except ResponseError as error:
            return str(error)

    def hotel_search(self, city_code, adults=1, check_in_date=None, check_out_date=None, 
                           price_range=None, ratings=None, top_k=10):
        try:
            params = {
                'cityCode': city_code,
                'adults': adults
            }

            if check_in_date and check_out_date:
                params['checkInDate'] = check_in_date
                params['checkOutDate'] = check_out_date
            if price_range:
                params['priceRange'] = price_range
            if ratings:
                params['ratings'] = ratings

            response = self.amadeus.shopping.hotel_offers_search.get(**params)
            return response.data[:top_k]
        except ResponseError as error:
            return str(error)

    def hotel_list_by_city(self, city_code, top_k=10):
        try:
            response = self.amadeus.reference_data.locations.hotels.by_city.get(cityCode=city_code)
            return response.data[:top_k]
        except ResponseError as error:
            return str(error)

    def travel_recommendations(self, city_codes, top_k=10):
        try:
            response = self.amadeus.reference_data.recommended_locations.get(cityCodes=city_codes)
            return response.data[:top_k]
        except ResponseError as error:
            return str(error)

    def airport_routes(self, departure_airport_code, top_k=10):
        try:
            response = self.amadeus.airport.direct_destinations.get(
                departureAirportCode=departure_airport_code
            )
            return response.data[:top_k]
        except ResponseError as error:
            return str(error)

    def transfer_booking(self, booking_request):
        try:
            response = self.amadeus.ordering.transfer_orders.post(body=booking_request)
            return response.data
        except ResponseError as error:
            return str(error)

    def flight_most_booked_destinations(self, origin_city_code, period=None, top_k=10):
        try:
            params = {'originCityCode': origin_city_code}
            if period:
                params['period'] = period

            response = self.amadeus.travel.analytics.air_traffic.booked.get(**params)
            return response.data[:top_k]
        except ResponseError as error:
            return str(error)

    def flight_most_traveled_destinations(self, origin_city_code, period=None, top_k=10):
        try:
            params = {'originCityCode': origin_city_code}
            if period:
                params['period'] = period

            response = self.amadeus.travel.analytics.air_traffic.traveled.get(**params)
            return response.data[:top_k]
        except ResponseError as error:
            return str(error)

amadeus_api = AmadeusAPI()

In [2]:
print(amadeus_api.hotel_list_by_city("HAN"))

[{'chainCode': 'AH', 'iataCode': 'HAN', 'dupeId': 502322079, 'name': 'HOTEL CONTENT A LA CARTE 1AH', 'hotelId': 'AHHAN001', 'geoCode': {'latitude': 21.02395, 'longitude': 105.83401}, 'address': {'countryCode': 'VN'}, 'lastUpdate': '2025-05-15T05:12:05'}, {'chainCode': 'AH', 'iataCode': 'HAN', 'dupeId': 502183886, 'name': 'JSON TEST 1AH', 'hotelId': 'AHHAN444', 'geoCode': {'latitude': 21.02395, 'longitude': 105.83401}, 'address': {'countryCode': 'VN'}, 'lastUpdate': '2023-06-15T10:30:36'}, {'chainCode': 'AZ', 'iataCode': 'HAN', 'dupeId': 700072044, 'name': 'SOMERSET WEST LAKE', 'hotelId': 'AZHAN131', 'geoCode': {'latitude': 21.04417, 'longitude': 105.81844}, 'address': {'countryCode': 'VN'}, 'lastUpdate': '2023-06-15T10:04:40'}, {'chainCode': 'AZ', 'iataCode': 'HAN', 'dupeId': 700099680, 'name': 'SOMERSET HOA BINH', 'hotelId': 'AZHAN186', 'geoCode': {'latitude': 21.04594, 'longitude': 105.79885}, 'address': {'countryCode': 'VN'}, 'lastUpdate': '2023-06-15T10:20:23'}, {'chainCode': 'CH',

# make the tool for the agent to use

In [3]:
def trip_information(
    originLocationCode: str,
    destinationLocationCode: str,
    departure_date: str,
    return_date: str,
    adults: int,
    travel_class: str,
    budget: int,
) -> Dict[str, Any]:
    """
    Collect comprehensive trip information including flights, hotels, activities, and recommendations.
    Returns all data in JSON format.
    """
    response_data = {
        "trip_request": {
            "originLocationCode": originLocationCode,
            "destinationLocationCode": destinationLocationCode,
            "departure_date": departure_date,
            "return_date": return_date,
            "adults": adults,
            "travel_class": travel_class,
            "budget": budget,
        },
        "flights": {},
        "hotels": {},
        "activities": {},
        "recommendations": {}
    }
    top_k = 2
    try:
        flight_offers = amadeus_api.flight_offers_search(
            origin=originLocationCode, 
            destination=destinationLocationCode, 
            departure_date=departure_date, 
            adults=adults, 
            return_date=return_date, 
            max_price=budget, 
            travel_class=travel_class, 
            top_k=top_k
        )
        response_data["flights"]["offers"] = flight_offers
    except Exception as e:
        response_data["flights"]["offers_error"] = str(e)

    try:
        cheapest_dates = amadeus_api.flight_cheapest_date_search(
            origin=originLocationCode,
            destination=destinationLocationCode,
            top_k=top_k
        )
        response_data["flights"]["cheapest_dates"] = cheapest_dates
    except Exception as e:
        response_data["flights"]["cheapest_dates_error"] = str(e)

    try:
        flight_inspiration = amadeus_api.flight_inspiration_search(
            origin=originLocationCode,
            top_k=top_k
        )
        response_data["flights"]["inspiration"] = flight_inspiration
    except Exception as e:
        response_data["flights"]["inspiration_error"] = str(e)

    try:
        most_booked = amadeus_api.flight_most_booked_destinations(
            origin_city_code=originLocationCode[:3],
            top_k=top_k
        )
        response_data["recommendations"]["most_booked"] = most_booked
    except Exception as e:
        response_data["recommendations"]["most_booked_error"] = str(e)

    try:
        most_traveled = amadeus_api.flight_most_traveled_destinations(
            origin_city_code=originLocationCode[:3], 
            top_k=top_k
        )
        response_data["recommendations"]["most_traveled"] = most_traveled
    except Exception as e:
        response_data["recommendations"]["most_traveled_error"] = str(e)

    try:
        hotel_offers = amadeus_api.hotel_search(
            city_code=destinationLocationCode[:3],
            adults=adults,
            check_in_date=departure_date,
            check_out_date=return_date,
            price_range=budget,
            top_k=top_k
        )
        response_data["hotels"]["offers"] = hotel_offers
    except Exception as e:
        response_data["hotels"]["offers_error"] = str(e)
    
    try:
        hotel_list = amadeus_api.hotel_list_by_city(
            city_code=destinationLocationCode[:3],
            top_k=top_k
        )
        response_data["hotels"]["list"] = hotel_list
    except Exception as e:
        response_data["hotels"]["list_error"] = str(e)

    try:
        travel_recs = amadeus_api.travel_recommendations(
            city_codes=destinationLocationCode[:3],
            top_k=top_k
        )
        response_data["recommendations"]["similar_destinations"] = travel_recs
    except Exception as e:
        response_data["recommendations"]["similar_destinations_error"] = str(e)
    
    try:
        routes = amadeus_api.airport_routes(
            departure_airport_code=originLocationCode,
            top_k=top_k
        )
        response_data["flights"]["routes"] = routes
    except Exception as e:
        response_data["flights"]["routes_error"] = str(e)

    return response_data

# Define the agent

In [4]:
DEFAULT_TRIP_AGENT_PROMPT = """# FRIENDLY TRAVEL COMPANION

## Your Role
You're a helpful travel companion who makes trip planning effortless. Your main goal is to minimize back-and-forth while creating personalized travel plans.

## Understanding IATA Airport Codes
- The originLocationCode and destinationLocationCode parameters require IATA airport codes
- IATA codes are the standardized 3-letter airport codes defined by the International Air Transport Association
- For any city mentioned, identify and use the appropriate IATA code for its main airport
- If multiple airports serve a city, use the major international airport unless specified otherwise

## Reading User Intent
- When users are specific, use their exact preferences
- When users are vague or say "just plan it for me," proactively make reasonable suggestions and proceed
- If a user says not to ask questions, respect that completely - make your best suggestions and move forward

## Parameter Collection & Suggestion
When users don't specify parameters, SUGGEST values instead of asking questions:

1. **Essential Flight Parameters:**
   - originLocationCode: 3-letter IATA airport code for departure location
   - destinationLocationCode: 3-letter IATA airport code for arrival location
   - departure_date: Format as YYYY-MM-DD
   - return_date: Format as YYYY-MM-DD
   - adults: Number of travelers
   - travel_class: "ECONOMY", "PREMIUM_ECONOMY", "BUSINESS", or "FIRST"
   - budget: Total trip budget in whole numbers

2. **How to Handle Missing Information:**
   - For IATA codes: Convert city names to their corresponding airport codes 
   - For dates: Suggest dates 2-4 weeks in the future for a 5-7 day trip
   - For destinations: Suggest based on interests or season (beaches in summer, skiing in winter)
   - For budget: Suggest ranges based on destination and class ($1000-$2000 for economy domestic, etc.)
   - For class: Default to "ECONOMY" unless user hints at luxury preferences

## When to Call plan_trip_function
Call the function when:
1. You have enough information to make meaningful suggestions for all parameters
2. You've converted any city names to their correct IATA airport codes
3. The user has explicitly approved your suggestions OR
4. The user has asked you to "just plan it" without further questions

## Creating the Travel Plan
After receiving search results:
1. Transform the data into a clear, structured travel plan
2. Include a daily itinerary with specific activities
3. Highlight practical details like transportation options and costs
4. Present the information in an easy-to-read format with clear sections

## Language and Cultural Adaptation
- Always respond in the same language the user is using
- Adapt recommendations to reflect the user's cultural context when possible
- Include culturally relevant suggestions for activities and dining
- Consider cultural preferences when suggesting accommodations and travel styles
- Use local currency formats and measurement units familiar to the user
- Include local etiquette tips and cultural insights relevant to their destination
- If the user is traveling to a region with a different language, include key useful phrases

## Response Style
- Be concise but thorough
- Use a warm, conversational tone
- Present information in scannable sections
- Include helpful local insights and travel tips

Remember: Your value is in making decisions and suggestions when users don't want to provide all details. When they say "just plan it" - do exactly that with your best judgment rather than asking more questions.
"""

In [10]:
from google.adk.agents import Agent
from google.genai import types
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from pydantic import BaseModel, Field

In [12]:
APP_NAME = "agent_app"
USER_ID = "test_user_456"
SESSION_ID_TOOL_AGENT = "session_tool_agent_xyz"
MODEL_NAME = "gemini-2.0-flash"

In [9]:
root_agent = Agent(
    name="root_agent",
    model="gemini-2.0-flash",
    instruction=DEFAULT_TRIP_AGENT_PROMPT,   
    tools=[
        trip_information
    ],
    generate_content_config=types.GenerateContentConfig(
        temperature=0.7
    )
)

In [15]:
# --- Set up Session Management and Runners ---
session_service = InMemorySessionService()
# Create separate sessions for clarity, though not strictly necessary if context is managed
session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID_TOOL_AGENT)

Session(id='session_tool_agent_xyz', app_name='agent_app', user_id='test_user_456', state={}, events=[], last_update_time=1747394492.6602707)

In [16]:
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service
)

In [25]:
async def call_agent_and_print(
    runner_instance: Runner,
    agent_instance: Agent,
    session_id: str,
    query_json: str
):
    """Sends a query to the specified agent/runner and prints results."""
    print(f"\n>>> Calling Agent: '{agent_instance.name}' | Query: {query_json}")

    user_content = types.Content(role='user', parts=[types.Part(text=query_json)])

    final_response_content = "No final response received."
    async for event in runner_instance.run_async(user_id=USER_ID, session_id=session_id, new_message=user_content):
        print(f"Event: {event}")
        if event.is_final_response() and event.content and event.content.parts:
            # For output_schema, the content is the JSON string itself
            final_response_content = event.content.parts[0].text

    print(f"<<< Agent '{agent_instance.name}' Response: {final_response_content}")

    current_session = session_service.get_session(app_name=APP_NAME,
                                                  user_id=USER_ID,
                                                  session_id=session_id)
    print(f"Current Session: {current_session}")
    print(f"Output Key: {agent_instance.output_key}")
    stored_output = current_session.state.get(agent_instance.output_key)

    # Pretty print if the stored output looks like JSON (likely from output_schema)
    print(f"--- Session State ['{agent_instance.output_key}']: ", end="")
    try:
        # Attempt to parse and pretty print if it's JSON
        parsed_output = json.loads(stored_output)
        print(json.dumps(parsed_output, indent=2))
    except (json.JSONDecodeError, TypeError):
         # Otherwise, print as string
        print(stored_output)
    print("-" * 30)
    
# --- Run Interactions ---
async def main():
    print("--- Testing Agent with Tool ---")
    await call_agent_and_print(runner, root_agent, SESSION_ID_TOOL_AGENT, 'Ok tuyệt, build plan cho tôi đi')


In [19]:
# --- Run Interactions ---
async def main():
    print("--- Testing Agent with Tool ---")
    await call_agent_and_print(runner, root_agent, SESSION_ID_TOOL_AGENT, 'Tôi sẽ bay từ Hà Nội vào khoảng mùng 1 tháng 6 năm 2025, tôi muốn đi khoảng 1 tuần cùng 3 người, hạng vé economy, ngân sách 10k USD')
await main()

--- Testing Agent with Tool ---

>>> Calling Agent: 'root_agent' | Query: Tôi sẽ bay từ Hà Nội vào khoảng mùng 1 tháng 6 năm 2025, tôi muốn đi khoảng 1 tuần cùng 3 người, hạng vé economy, ngân sách 10k USD
<<< Agent 'root_agent' Response: Tuyệt vời! Với những thông tin bạn cung cấp, tôi sẽ lên kế hoạch cho chuyến đi đến New York từ Hà Nội (HAN) vào khoảng mùng 1 tháng 6 năm 2025, kéo dài khoảng 1 tuần cho 3 người, hạng vé economy và ngân sách 10.000 USD.

*   **Điểm khởi hành:** HAN
*   **Điểm đến:** NYC
*   **Ngày đi:** 2025-06-01
*   **Ngày về:** 2025-06-08
*   **Số lượng người lớn:** 3
*   **Hạng vé:** ECONOMY
*   **Ngân sách:** 10000

Bạn có muốn tôi tiến hành lên kế hoạch với những thông tin này không?
--- Session State ['None']: None
------------------------------


In [20]:
async def main():
    print("--- Testing Agent with Tool ---")
    await call_agent_and_print(runner, root_agent, SESSION_ID_TOOL_AGENT, 'Có, build plan cho tôi')
await main()

--- Testing Agent with Tool ---

>>> Calling Agent: 'root_agent' | Query: Có, build plan cho tôi


<<< Agent 'root_agent' Response: Tuyệt vời! Dựa trên thông tin bạn cung cấp, đây là kế hoạch sơ bộ cho chuyến đi của bạn:

**Chuyến bay:**

*   **Hãng hàng không:** Korean Air (KE)
*   **Giá vé (cho 3 người):** Khoảng 2845.83 USD (948.61 USD/người)
*   **Hành trình:**
    *   **Đi:**
        *   HAN (Hà Nội) -> ICN (Seoul)
        *   ICN (Seoul) -> JFK (New York)
    *   **Về:**
        *   JFK (New York) -> ICN (Seoul)
        *   ICN (Seoul) -> HAN (Hà Nội)

**Lưu ý quan trọng:**

*   Đây là giá vé máy bay tham khảo và có thể thay đổi tùy thuộc vào thời điểm đặt vé.
*   Bạn nên đặt vé máy bay càng sớm càng tốt để có giá tốt nhất.

**Khách sạn:**

Tôi thấy một vài gợi ý khách sạn ở khu vực Brooklyn và Harlem:

*   **Aloft New York Brooklyn:** Khách sạn hiện đại ở Brooklyn.
*   **Aloft Harlem:** Khách sạn ở khu vực Harlem sôi động.

**Lịch trình gợi ý:**

*   **Ngày 1:** Đến New York, nhận phòng khách sạn, khám phá khu vực xung quanh.
*   **Ngày 2:** Tham quan các địa điểm nổi tiếng n

In [24]:
await main()

--- Testing Agent with Tool ---

>>> Calling Agent: 'root_agent' | Query: Tôi muốn đi du lịch Nhật Bản
Event: content=Content(parts=[Part(video_metadata=None, thought=None, code_execution_result=None, executable_code=None, file_data=None, function_call=None, function_response=None, inline_data=None, text='Tuyệt vời! Nhật Bản là một điểm đến tuyệt vời. Để tôi có thể lên kế hoạch sơ bộ cho bạn, tôi xin phép đề xuất một vài thông tin sau nhé:\n\n*   **Điểm khởi hành:** Tôi đề xuất bạn bay từ Hà Nội (HAN).\n*   **Thời gian:** Tôi đề xuất bạn đi vào khoảng 3 tuần tới, từ ngày 7/6/2024 đến ngày 14/6/2024 (7 ngày).\n*   **Số lượng người:** Tôi đề xuất là 2 người lớn.\n*   **Hạng vé:** Tôi đề xuất hạng phổ thông (ECONOMY).\n*   **Ngân sách:** Tôi đề xuất ngân sách khoảng 3000 USD cho chuyến đi này.\n\nVới những thông tin này, tôi có thể bắt đầu lên kế hoạch sơ bộ cho chuyến đi Nhật Bản của bạn. Bạn thấy thế nào?')], role='model') grounding_metadata=None partial=None turn_complete=None error_co

In [26]:
await main()

--- Testing Agent with Tool ---

>>> Calling Agent: 'root_agent' | Query: Ok tuyệt, build plan cho tôi đi


Event: content=Content(parts=[Part(video_metadata=None, thought=None, code_execution_result=None, executable_code=None, file_data=None, function_call=FunctionCall(id='adk-d730e4f8-b667-45a4-96e6-14e3fde31988', args={'departure_date': '2024-06-07', 'travel_class': 'ECONOMY', 'return_date': '2024-06-14', 'budget': 3000, 'destinationLocationCode': 'TYO', 'originLocationCode': 'HAN', 'adults': 2}, name='trip_information'), function_response=None, inline_data=None, text=None)], role='model') grounding_metadata=None partial=None turn_complete=None error_code=None error_message=None interrupted=None invocation_id='e-ed6f0cd1-8264-492b-a0c6-9a945ccd544b' author='root_agent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}) long_running_tool_ids=set() branch=None id='W9E2HB75' timestamp=1747395774.229644
Event: content=Content(parts=[Part(video_metadata=None, thought=None, code_execution_result=None